<a href="https://colab.research.google.com/github/shivamkumar359/03-rd-SEM--ASSIGNMENTS/blob/main/PDS/W12/Q2/ASSIGNMENT_Q2_1_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# TASK 2: TRACE AND ANALYZE KRUSKAL MINIMUM SPANNING TREE
# IIT Jodhpur BSc
# ============================================================

import os
import sys
import textwrap
import subprocess

# Create submission directory
os.makedirs("Kruskal_Task2", exist_ok=True)


# ============================================================
# PART 1: CREATE kruskal_mst.py
# ============================================================

kruskal_code = r'''
# ============================================================
# KRUSKAL'S MINIMUM SPANNING TREE ALGORITHM
# ============================================================

import matplotlib.pyplot as plt


# ------------------------------------------------------------
# GRAPH REPRESENTATION
# ------------------------------------------------------------

vertices = ['A', 'B', 'C', 'D', 'E', 'F']

# Edge format: (weight, vertex1, vertex2)

edges = [
    (1, 'D', 'E'),
    (2, 'A', 'D'),
    (2, 'C', 'F'),
    (3, 'A', 'B'),
    (4, 'B', 'E'),
    (5, 'B', 'C'),
    (6, 'E', 'F')
]


# ------------------------------------------------------------
# DISJOINT SET UNION (DSU)
# CYCLE DETECTION HELPER
# ------------------------------------------------------------

class DisjointSet:

    def __init__(self, vertices):

        self.parent = {}
        self.rank = {}

        for vertex in vertices:
            self.parent[vertex] = vertex
            self.rank[vertex] = 0

    # Find representative of a set
    # Path compression is used for efficiency

    def find(self, vertex):

        if self.parent[vertex] != vertex:
            self.parent[vertex] = self.find(
                self.parent[vertex]
            )

        return self.parent[vertex]

    # Merge two sets
    # Union by rank is used

    def union(self, u, v):

        root_u = self.find(u)
        root_v = self.find(v)

        if root_u == root_v:
            return False

        if self.rank[root_u] < self.rank[root_v]:

            self.parent[root_u] = root_v

        elif self.rank[root_u] > self.rank[root_v]:

            self.parent[root_v] = root_u

        else:

            self.parent[root_v] = root_u
            self.rank[root_u] += 1

        return True


# ------------------------------------------------------------
# KRUSKAL'S ALGORITHM
# ------------------------------------------------------------

def kruskal_mst(vertices, edges):

    # Initialize Disjoint Set
    ds = DisjointSet(vertices)

    # Sort edges in ascending order of weight
    sorted_edges = sorted(
        edges,
        key=lambda edge: edge[0]
    )

    mst = []
    rejected_edges = []

    total_weight = 0

    print("=" * 75)
    print("       KRUSKAL'S MINIMUM SPANNING TREE ALGORITHM")
    print("=" * 75)

    print("\\nOriginal Edges:")
    for weight, u, v in edges:
        print(f"{u} -- {v} : {weight}")

    print("\\nEdges Sorted in Ascending Order:")
    for weight, u, v in sorted_edges:
        print(f"{u} -- {v} : {weight}")

    print("\\n" + "=" * 75)
    print("EDGE SELECTION AND CYCLE DETECTION")
    print("=" * 75)

    # Process edges in sorted order
    for weight, u, v in sorted_edges:

        print(f"\\nChecking Edge: {u} -- {v} | Weight: {weight}")

        root_u = ds.find(u)
        root_v = ds.find(v)

        # Check whether both vertices belong to the same set

        if root_u != root_v:

            # No cycle: accept the edge
            ds.union(u, v)

            mst.append((u, v, weight))

            total_weight += weight

            print("STATUS: ACCEPTED")
            print("Reason: No cycle is formed.")
            print("Added to Minimum Spanning Tree.")

        else:

            # Cycle detected: reject the edge
            rejected_edges.append((u, v, weight))

            print("STATUS: REJECTED")
            print("Reason: Adding this edge creates a cycle.")

        print("Current MST Weight:", total_weight)

        # MST is complete after V-1 accepted edges
        if len(mst) == len(vertices) - 1:
            break

    # --------------------------------------------------------
    # FINAL MST RESULTS
    # --------------------------------------------------------

    print("\\n" + "=" * 75)
    print("FINAL MINIMUM SPANNING TREE")
    print("=" * 75)

    print("\\nAccepted Edges:")

    for u, v, weight in mst:
        print(f"{u} -- {v} : {weight}")

    print("\\nRejected Cyclic Edges:")

    for u, v, weight in rejected_edges:
        print(f"{u} -- {v} : {weight}")

    print("\\nTotal MST Weight:", total_weight)

    print("Number of Vertices:", len(vertices))
    print("Number of MST Edges:", len(mst))

    print("\\nMST Construction Successful!")

    return mst, rejected_edges, total_weight


# ------------------------------------------------------------
# VISUALIZE AND GENERATE MST PDF
# ------------------------------------------------------------

def generate_mst_pdf(vertices, mst, rejected_edges):

    # Fixed positions for a clear graph layout
    positions = {
        'A': (0, 2),
        'B': (1, 2),
        'C': (3, 2),
        'D': (0, 0),
        'E': (1, 0),
        'F': (3, 0)
    }

    fig, ax = plt.subplots(figsize=(11, 7))

    # Draw rejected edges as dashed lines
    for u, v, weight in rejected_edges:

        x1, y1 = positions[u]
        x2, y2 = positions[v]

        ax.plot(
            [x1, x2],
            [y1, y2],
            linestyle='--',
            color='red',
            linewidth=1.8,
            alpha=0.7,
            zorder=1
        )

        # Label rejected edge
        mid_x = (x1 + x2) / 2
        mid_y = (y1 + y2) / 2

        ax.text(
            mid_x,
            mid_y + 0.12,
            str(weight),
            color='red',
            fontsize=11,
            fontweight='bold',
            ha='center'
        )

    # Draw accepted MST edges
    for u, v, weight in mst:

        x1, y1 = positions[u]
        x2, y2 = positions[v]

        ax.plot(
            [x1, x2],
            [y1, y2],
            color='green',
            linewidth=3,
            zorder=2
        )

        mid_x = (x1 + x2) / 2
        mid_y = (y1 + y2) / 2

        ax.text(
            mid_x + 0.08,
            mid_y + 0.12,
            str(weight),
            color='darkgreen',
            fontsize=12,
            fontweight='bold',
            ha='center'
        )

    # Draw vertices
    for vertex, (x, y) in positions.items():

        ax.scatter(
            x, y,
            s=1100,
            color='lightblue',
            edgecolors='black',
            linewidths=2,
            zorder=3
        )

        ax.text(
            x, y,
            vertex,
            fontsize=16,
            fontweight='bold',
            ha='center',
            va='center',
            zorder=4
        )

    ax.set_title(
        "Kruskal's Minimum Spanning Tree",
        fontsize=18,
        fontweight='bold',
        pad=20
    )

    ax.text(
        0.5, 1.02,
        "Minimum Total Weight = 13",
        transform=ax.transAxes,
        fontsize=14,
        ha='center',
        color='darkgreen',
        fontweight='bold'
    )

    ax.text(
        0.5, -0.08,
        "Green Solid Edges: Accepted MST Edges    |    "
        "Red Dashed Edges: Rejected Cyclic Edges",
        transform=ax.transAxes,
        fontsize=10,
        ha='center'
    )

    ax.set_xlim(-0.8, 3.8)
    ax.set_ylim(-0.8, 2.8)

    ax.axis('off')

    plt.tight_layout()

    # Save diagram as PDF
    plt.savefig(
        "mst_structure.pdf",
        format="pdf",
        bbox_inches="tight"
    )

    plt.close()

    print("\\nPDF Generated Successfully: mst_structure.pdf")


# ------------------------------------------------------------
# MAIN PROGRAM EXECUTION
# ------------------------------------------------------------

if __name__ == "__main__":

    mst, rejected_edges, total_weight = kruskal_mst(
        vertices,
        edges
    )

    generate_mst_pdf(
        vertices,
        mst,
        rejected_edges
    )
'''

# Save Python script
py_file = "Kruskal_Task2/kruskal_mst.py"

with open(py_file, "w") as file:
    file.write(textwrap.dedent(kruskal_code))

print("Created:", py_file)


# ============================================================
# PART 2: EXECUTE KRUSKAL PROGRAM
# ============================================================

print("\n\nEXECUTING KRUSKAL MST PROGRAM...\n")

result = subprocess.run(
    [sys.executable, py_file],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:

    print("EXECUTION ERROR:")
    print(result.stderr)

else:

    print("\nPROGRAM EXECUTED SUCCESSFULLY!")


# ============================================================
# PART 3: MOVE GENERATED PDF INTO SUBMISSION DIRECTORY
# ============================================================

import shutil

if os.path.exists("mst_structure.pdf"):

    shutil.move(
        "mst_structure.pdf",
        "Kruskal_Task2/mst_structure.pdf"
    )

    print("PDF moved to submission directory.")


# ============================================================
# PART 4: DISPLAY GENERATED MST PDF
# ============================================================

from IPython.display import display, IFrame

pdf_path = "Kruskal_Task2/mst_structure.pdf"

if os.path.exists(pdf_path):

    print("\nDISPLAYING MST STRUCTURE PDF:\n")

    display(
        IFrame(
            src=pdf_path,
            width=850,
            height=550
        )
    )


# ============================================================
# PART 5: DOWNLOAD SUBMISSION FILES
# ============================================================

from google.colab import files

print("\n" + "=" * 75)
print("TASK 2 SUBMISSION FILES")
print("=" * 75)

print("\n1. kruskal_mst.py")
files.download("Kruskal_Task2/kruskal_mst.py")

print("\n2. mst_structure.pdf")
files.download("Kruskal_Task2/mst_structure.pdf")

print("\nTASK 2 COMPLETED SUCCESSFULLY!")

Created: Kruskal_Task2/kruskal_mst.py


EXECUTING KRUSKAL MST PROGRAM...

       KRUSKAL'S MINIMUM SPANNING TREE ALGORITHM
\nOriginal Edges:
D -- E : 1
A -- D : 2
C -- F : 2
A -- B : 3
B -- E : 4
B -- C : 5
E -- F : 6
\nEdges Sorted in Ascending Order:
D -- E : 1
A -- D : 2
C -- F : 2
A -- B : 3
B -- E : 4
B -- C : 5
E -- F : 6
\n===========================================================================
EDGE SELECTION AND CYCLE DETECTION
\nChecking Edge: D -- E | Weight: 1
STATUS: ACCEPTED
Reason: No cycle is formed.
Added to Minimum Spanning Tree.
Current MST Weight: 1
\nChecking Edge: A -- D | Weight: 2
STATUS: ACCEPTED
Reason: No cycle is formed.
Added to Minimum Spanning Tree.
Current MST Weight: 3
\nChecking Edge: C -- F | Weight: 2
STATUS: ACCEPTED
Reason: No cycle is formed.
Added to Minimum Spanning Tree.
Current MST Weight: 5
\nChecking Edge: A -- B | Weight: 3
STATUS: ACCEPTED
Reason: No cycle is formed.
Added to Minimum Spanning Tree.
Current MST Weight: 8
\nChecking Edge: 


TASK 2 SUBMISSION FILES

1. kruskal_mst.py


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


2. mst_structure.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


TASK 2 COMPLETED SUCCESSFULLY!
